<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/Module4_Labs/Lab11.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lab 11 — Pauli Operators, Tensor Products, and Pauli Rotations
**Quantum Optimization and Simulation — VQE Laboratory Series**

Build simple two-qubit operators, connect their matrices to circuits, and run those circuits on `AerSimulator`.

**Suggested use:** brief instructor demonstration followed by guided or independent work.

**Notebook style:** Most code is supplied. Focus on what each gate does rather than memorizing matrix algebra.

> Qiskit displays measured bitstrings as `q_(n-1)...q_0`.


## Learning objectives
1. Build two-qubit Pauli operators with `np.kron`.
2. Interpret \(R_{ZZ}\) as a phase-changing operation.
3. See how \(R_{YX}R_{XY}\) can mix \(|01\rangle\) and \(|10\rangle\).
4. Connect matrix expressions to Qiskit circuits.
5. Run the circuits with `AerSimulator`.


In [ ]:
# Run once in a fresh Google Colab session.
%pip -q install pylatexenc matplotlib
%pip -q install "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-algorithms~=0.4" "qiskit-nature~=0.8"

## Part A. Two qubits: the tensor (Kronecker) product

Two qubits live in a **4**-dimensional space with basis
$|00\rangle, |01\rangle, |10\rangle, |11\rangle$. An operator acting on both qubits is a
4×4 matrix built with $\otimes$ (`np.kron`):

$$ (A \otimes B)\,|q_0 q_1\rangle \;=\; A|q_0\rangle \otimes B|q_1\rangle .$$

`np.kron(A, B)` tiles a copy of `B` scaled by every entry of `A`. That is all it is.


In [ ]:
import numpy as np

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

X  = np.array([[0, 1], [1, 0]], dtype=complex)
Y  = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z  = np.array([[1, 0], [0, -1]], dtype=complex)

YX = np.kron(Y, X)
XY = np.kron(X, Y)
ZZ = np.kron(Z, Z)

print("Y (x) X =\n", YX) #.real)
print("\nX (x) Y =\n", XY) #.real,)
print("\nZ (x) Z =\n", ZZ) #.real)
print("\nYX & XY: all imaginary, ZZ: all real\n")


## Part B(1). Rzz: Put it in an exponent

For any Hermitian $P$ with $P^2 = I$ (every Pauli string qualifies),

$$ e^{-i\frac{\theta}{2}P} \;=\; \cos\!\Big(\frac{\theta}{2}\Big) I \;-\; i\,\sin\!\Big(\frac{\theta}{2}\Big) P ,$$

and this **is** unitary for every real $\theta$. It is the matrix version of Euler's
formula $e^{-i\phi} = \cos\phi - i\sin\phi$; the series works out because
$P^2 = I$ collapses all the even powers.

Two useful readings of the same object:
* $\theta$ is a **knob**. At $\theta=0$ you get the identity; turning it continuously
  deforms the state.
* Wherever $P$ has eigenvalue $+1$ you pick up phase $e^{-i\theta/2}$; where it has
  eigenvalue $-1$ you pick up $e^{+i\theta/2}$. A Pauli exponential is a
  **phase rotation conditioned on the eigenvalue of $P$**.


In [ ]:
def pauli_exp(P, theta):
    '''exp(-i theta/2 P) computed from the closed form, for P Hermitian with P^2 = I.'''
    n = P.shape[0]
    return np.cos(theta/2) * np.eye(n) - 1j * np.sin(theta/2) * P

b00 = np.array([1,0,0,0], dtype=complex)
b01 = np.array([0,1,0,0], dtype=complex)
b10 = np.array([0,0,1,0], dtype=complex)
b11 = np.array([0,0,0,1], dtype=complex)

theta = 0.7
RZZ = pauli_exp(ZZ, theta)
print("RZZ(theta) =\n", np.round(RZZ, 3))
print("\nRZZ|00> =", np.round(RZZ@b00, 3))
print("RZZ|01> =", np.round(RZZ@b01, 3))
print("RZZ|10> =", np.round(RZZ@b10, 3))
print("RZZ|11> =", np.round(RZZ@b11, 3))

##Part B(2). Rzz: From matrix to circuit: CNOT — R$_z(\theta)$ — CNOT

Hardware does not accept a 4×4 matrix. The standard compilation is

```
q0: ──■─────────────■──
      │             │
q1: ──⊕──[Rz(θ)]────⊕──
```

The first CNOT writes the **parity** $a\oplus b$ into qubit 1; `Rz` applies the
parity-dependent phase; the second CNOT undoes the bookkeeping. Three gates, one knob.

This matters practically: on IBM superconducting hardware `Rz` is a *virtual* gate — the
control electronics just shift the phase reference of later pulses — so it is essentially
free. The cost of this circuit is the two CNOTs.

NOTE:$R_{ZZ}$  alone can never mix configurations

In [ ]:
def rzz_circuit(theta):
    '''CNOT - Rz(theta) on q1 - CNOT.'''
    qc = QuantumCircuit(2, name="RZZ")
    qc.cx(0, 1)
    qc.rz(theta, 1)
    qc.cx(0, 1)
    return qc

theta = 0.7
qc = rzz_circuit(theta)
print(qc.draw(output="text"))

### Run the $R_{ZZ}$ circuit on AerSimulator

The matrix calculation above is useful for understanding the gate. Now run the actual Qiskit circuit.

We start from $|01\rangle$. Because $R_{ZZ}$ changes **phase** rather than probabilities, we use Aer’s statevector simulator so that the phase is visible.


In [ ]:
backend = AerSimulator(method="statevector")

qc_aer = QuantumCircuit(2)
qc_aer.x(0)                     # Qiskit display: |01>
qc_aer.compose(rzz_circuit(theta), inplace=True)
qc_aer.save_statevector()

tqc_aer = transpile(qc_aer, backend)
result_aer = backend.run(tqc_aer).result()
sv_aer = result_aer.get_statevector(tqc_aer)

print("AerSimulator statevector:")
print(np.round(np.asarray(sv_aer), 3))


## Part C(1). $R_{YX}R_{XY}$: Put it in an exponent


$$ \mathrm{R}_{YX}(\theta)\,\mathrm{R}_{XY}(-\theta)
\;=\; e^{-i\frac{\theta}{2}Y_0X_1}\, e^{i\frac{\theta}{2}X_0Y_1}
\;=\; e^{-i\frac{\theta}{2}(Y_0X_1 - X_0Y_1)} .$$

We want a circuit that moves amplitude between $|10\rangle$ and $|01\rangle$ — an electron hopping between the bonding and antibonding configuration — **without** touching $|00\rangle$ and $|11\rangle$. This lab builds the answer:

(The last equality holds because $Y_0X_1$ and $X_0Y_1$ **commute**.)

In [ ]:
theta = 0.7
Ryx = pauli_exp(YX, theta)
Rxy = pauli_exp(XY, -1*theta)
RyxRxy = Ryx@Rxy
print("Ryx(theta) =\n", np.round(Ryx, 3))
print("Rxy(-theta) =\n", np.round(Rxy, 3))
print("Ryx(theta)Rxy(-theta) =\n", np.round(RyxRxy, 3))
print("\nRyxRxy|00> =", np.round(RyxRxy@b00, 3))
print("RyxRxy|01> =", np.round(RyxRxy@b01, 3))
print("RyxRxy|10> =", np.round(RyxRxy@b10, 3))
print("RyxRxy|11> =", np.round(RyxRxy@b11, 3))


*Read the output carefully: $R_{YX}R_{XY}$ mixes $|10\rangle$ and $|01\rangle$ but it leaves $|00\rangle$ and $|11\rangle$ unaffected. In other words,

$R_{YX}R_{XY} |00\rangle = |00\rangle$

$R_{YX}R_{XY} |01\rangle = cos\theta\; |01\rangle-i\;sin\theta\;|10\rangle$

$R_{YX}R_{XY} |10\rangle = -i\; sin\theta\; |01\rangle+cos\theta\;|10\rangle$

$R_{YX}R_{XY} |11\rangle = |11\rangle$


##Part C(2). $R_{YX}R_{XY}$: From matrix to circuit

Hardware does not accept a 4×4 matrix. The standard compilation is

$R_{YX}(\theta)$
```
q0: ─S+─H─■─────────────■──H─S─
          │             │
q1: ────H─⊕──[Rz(θ)]───⊕─H───
```

$R_{XY}(-\theta)$
```
q0: ────H─■──────────────■──H───
          │              │
q1: ─S+─H─⊕──[Rz(-θ)]───⊕─H─S──
```

###NOTE: Basis changes: $H$ turns $Z$ into $X$

The Hadamard swaps the roles of the $x$ and $z$ axes on the Bloch sphere:

$$ H Z H = X, \qquad S H Z H S^\dagger = S X S^\dagger = Y .$$

This is the single most reusable trick in the course. It says: **anything you can do
along $Z$, you can do along $X$ or $Y$ by wrapping it in cheap one-qubit gates.**
You will use it again later to *measure* $\langle XX\rangle$ and $\langle YY\rangle$
on hardware that can only measure $Z$.


In [ ]:
def ryx_circuit(theta):
    '''CNOT - Rz(theta) on q1 - CNOT.'''
    qc = QuantumCircuit(2, name="RYX")
    qc.sdg(0)
    qc.h(0)
    qc.h(1)
    qc.cx(0, 1)
    qc.rz(theta, 1)
    qc.cx(0, 1)
    qc.h(0)
    qc.s(0)
    qc.h(1)
    return qc

def rxy_circuit(theta):
    '''CNOT - Rz(theta) on q1 - CNOT.'''
    qc = QuantumCircuit(2, name="RYX")
    qc.h(0)
    qc.sdg(1)
    qc.h(1)
    qc.cx(0, 1)
    qc.rz(theta, 1)
    qc.cx(0, 1)
    qc.h(0)
    qc.h(1)
    qc.s(1)
    return qc

theta = 0.7
qc = ryx_circuit(theta)
qc2 = rxy_circuit(-theta)
qc.compose(qc2, inplace=True)
print(qc.draw(output="text"))

### Exercise — why R$_{ZZ}$ alone can never mix configurations; How about R$_{YX}$R$_{XY}$?

Prepare the superposition $\frac{1}{\sqrt2}(|01\rangle + |10\rangle)$, apply
R$_{ZZ}(\theta)$ for several $\theta$, and print the probabilities. Repeat with R$_{YX}(\theta)$R$_{XY}(-\theta)$.

In [ ]:
psi0 = (b01 + b10) / np.sqrt(2)

print(f"{'theta':>6} | {'P(01)':>7} {'P(10)':>7} | amplitudes")
for theta in [0.0, 0.5, 1.0, 2.0, np.pi]:
    out = pauli_exp(ZZ, theta) @ psi0
    p = np.abs(out)**2
    print(f"{theta:6.2f} | {p[1]:7.4f} {p[2]:7.4f} | {np.round(out,3)}")

print("\nThe probabilities NEVER move. RZZ is a pure phase rotation:")
print("it changes how amplitudes will later interfere, but on its own it")
print("cannot move an electron from one configuration to another.")

print()
for theta in [0.0, 0.5, 1.0, 2.0, np.pi]:
    Ryx = pauli_exp(YX, theta)
    Rxy = pauli_exp(XY, -1*theta)
    out = Ryx@Rxy @ psi0
    p = np.abs(out)**2
    print(f"{theta:6.2f} | {p[1]:7.4f} {p[2]:7.4f} | {np.round(out,3)}")

print("\nThe probabilities move with RyxRxy.")


### AerSimulator check for the mixing circuit

Here we run the circuit form of \(R_{YX}(\theta)R_{XY}(-\theta)\) on Aer and inspect the final probabilities.


In [ ]:
qc_mix = QuantumCircuit(2)
qc_mix.x(0)                     # start from |01>
qc_mix.compose(ryx_circuit(theta), inplace=True)
qc_mix.compose(rxy_circuit(-theta), inplace=True)
qc_mix.save_statevector()

tqc_mix = transpile(qc_mix, backend)
result_mix = backend.run(tqc_mix).result()
sv_mix = np.asarray(result_mix.get_statevector(tqc_mix))

print("Final statevector:", np.round(sv_mix, 3))
print("Probabilities:", np.round(np.abs(sv_mix)**2, 3))
